Questo notebook fa parte di **Pensare in Python**, traduzione italiana non ufficiale di [*Think Python*, 3ª edizione](https://allendowney.github.io/ThinkPython/) di Allen B. Downey. Puoi leggere tutto il libro tradotto su <https://pensare-in-python.github.io/3a-edizione/>.

Le versioni stampata ed ebook dell'originale si possono acquistare da [Bookshop.org](https://bookshop.org/a/98697/9781098155438) e [Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325) (link di affiliazione dell'autore).

Se trovi utile questo libro, valuta di [sostenere i materiali didattici liberi e aperti](https://greenteapress.com/support/?book=tp3) di Green Tea Press.

In [ ]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');
download('https://github.com/ramalho/jupyturtle/releases/download/2024-03/jupyturtle.py');

import thinkpython

# Classi e oggetti

[Esegui questo notebook su Colab](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap16.ipynb)

A questo punto abbiamo definito classi e creato oggetti che rappresentano l'ora del giorno e il giorno dell'anno.
E abbiamo definito metodi che creano, modificano ed eseguono calcoli con questi oggetti.

In questo capitolo continueremo il nostro giro nella programmazione orientata agli oggetti (OOP) definendo classi che rappresentano oggetti geometrici, tra cui punti, linee, rettangoli e cerchi.
Scriveremo metodi che creano e modificano questi oggetti, e useremo il modulo `jupyturtle` per disegnarli.

Userò queste classi per illustrare alcuni argomenti della OOP, tra cui identità ed equivalenza degli oggetti, copia superficiale e profonda, e polimorfismo.

## Creare un punto

Nella grafica al computer una posizione sullo schermo viene spesso rappresentata con una coppia di coordinate in un piano `x`-`y`.
Per convenzione, il punto `(0, 0)` di solito rappresenta l'angolo in alto a sinistra dello schermo, e `(x, y)` rappresenta il punto che si trova `x` unità a destra e `y` unità in basso rispetto all'origine.
Rispetto al sistema di coordinate cartesiane che forse hai visto a lezione di matematica, l'asse `y` è capovolto.

Ci sono diversi modi in cui potremmo rappresentare un punto in Python:

-   Possiamo memorizzare le coordinate separatamente in due variabili, `x` e `y`.

-   Possiamo memorizzare le coordinate come elementi di una lista o di una tupla.

-   Possiamo creare un nuovo tipo per rappresentare i punti come oggetti.

Nella programmazione orientata agli oggetti, la cosa più idiomatica sarebbe creare un nuovo tipo.
Per farlo, inizieremo con una definizione di classe per `Point`.

In [ ]:
class Point:
    """Represents a point in 2-D space."""
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __str__(self):
        return f'Point({self.x}, {self.y})'

Il metodo `__init__` riceve le coordinate come parametri e le assegna agli attributi `x` e `y`.
Il metodo `__str__` restituisce una rappresentazione in stringa del `Point`.

Ora possiamo istanziare e visualizzare un oggetto `Point` così.

In [ ]:
start = Point(0, 0)
print(start)

Il diagramma seguente mostra lo stato del nuovo oggetto.

In [ ]:
from diagram import make_frame, make_binding

d1 = vars(start)
frame = make_frame(d1, name='Point', dy=-0.25, offsetx=0.18)
binding = make_binding('start', frame)

In [ ]:
from diagram import diagram, adjust

width, height, x, y = [1.41, 0.89, 0.26, 0.5]
ax = diagram(width, height)
bbox = binding.draw(ax, x, y)
#adjust(x, y, bbox)

Come al solito, un tipo definito dal programmatore è rappresentato da un riquadro con il nome del tipo all'esterno e gli attributi all'interno.

In generale, i tipi definiti dal programmatore sono mutabili, quindi possiamo scrivere un metodo come `translate` che riceve due numeri, `dx` e `dy`, e li aggiunge agli attributi `x` e `y`.

In [ ]:
%%add_method_to Point

    def translate(self, dx, dy):
        self.x += dx
        self.y += dy

Questa funzione trasla il `Point` da una posizione del piano a un'altra.
Se non vogliamo modificare un `Point` esistente, possiamo usare `copy` per copiare l'oggetto originale e poi modificare la copia.

In [ ]:
from copy import copy

end1 = copy(start)
end1.translate(300, 0)
print(end1)

Possiamo incapsulare questi passi in un altro metodo chiamato `translated`.

In [ ]:
%%add_method_to Point

    def translated(self, dx=0, dy=0):
        point = copy(self)
        point.translate(dx, dy)
        return point

Così come il metodo `sort` modifica una lista e la funzione `sorted` crea una nuova lista, ora abbiamo un metodo `translate` che modifica un `Point` e un metodo `translated` che ne crea uno nuovo.

Ecco un esempio:

In [ ]:
end2 = start.translated(0, 150)
print(end2)

Nella prossima sezione useremo questi punti per definire e disegnare una linea.

## Creare una linea

Ora definiamo una classe che rappresenta il segmento tra due punti.
Come al solito, inizieremo con un metodo `__init__` e un metodo `__str__`.

In [ ]:
class Line:
    def __init__(self, p1, p2):
        self.p1 = p1
        self.p2 = p2
        
    def __str__(self):
        return f'Line({self.p1}, {self.p2})'

Con questi due metodi possiamo istanziare e visualizzare un oggetto `Line` che useremo per rappresentare l'asse `x`.

In [ ]:
line1 = Line(start, end1)
print(line1)

Quando chiamiamo `print` e passiamo `line` come parametro, `print` invoca `__str__` su `line`.
Il metodo `__str__` usa una f-string per creare una rappresentazione in stringa di `line`.

La f-string contiene due espressioni tra parentesi graffe, `self.p1` e `self.p2`.
Quando queste espressioni vengono valutate, i risultati sono oggetti `Point`.
Poi, quando vengono convertiti in stringhe, viene invocato il metodo `__str__` della classe `Point`.

Ecco perché, quando visualizziamo una `Line`, il risultato contiene le rappresentazioni in stringa degli oggetti `Point`.

Il diagramma degli oggetti seguente mostra lo stato di questo oggetto `Line`.

In [ ]:
from diagram import Binding, Value, Frame

d1 = vars(line1.p1)
frame1 = make_frame(d1, name='Point', dy=-0.25, offsetx=0.17)

d2 = vars(line1.p2)
frame2 = make_frame(d2, name='Point', dy=-0.25, offsetx=0.17)

binding1 = Binding(Value('start'), frame1, dx=0.4)
binding2 = Binding(Value('end'), frame2, dx=0.4)
frame3 = Frame([binding1, binding2], name='Line', dy=-0.9, offsetx=0.4, offsety=-0.25)

binding = make_binding('line1', frame3)

In [ ]:
width, height, x, y = [2.45, 2.12, 0.27, 1.76]
ax = diagram(width, height)
bbox = binding.draw(ax, x, y)
#adjust(x, y, bbox)

Le rappresentazioni in stringa e i diagrammi degli oggetti sono utili per il debugging, ma lo scopo di questo esempio è generare grafica, non testo!
Quindi useremo il modulo `jupyturtle` per disegnare linee sullo schermo.

Come abbiamo fatto nel [capitolo 4](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap04.ipynb), useremo `make_turtle` per creare un oggetto `Turtle` e una piccola tela su cui può disegnare.
Per disegnare le linee, useremo due nuove funzioni del modulo `jupyturtle`:

* `jumpto`, che riceve due coordinate e sposta la `Turtle` nella posizione indicata senza disegnare una linea, e

* `moveto`, che sposta la `Turtle` dalla sua posizione attuale alla posizione indicata, e disegna un segmento tra le due.

Ecco come le importiamo.

In [ ]:
from jupyturtle import make_turtle, jumpto, moveto

Ed ecco un metodo che disegna una `Line`.

In [ ]:
%%add_method_to Line

    def draw(self):
        jumpto(self.p1.x, self.p1.y)
        moveto(self.p2.x, self.p2.y)

Per mostrare come si usa, creerò una seconda linea che rappresenta l'asse `y`.

In [ ]:
line2 = Line(start, end2)
print(line2)

E poi disegnerò gli assi.

In [ ]:
make_turtle()
line1.draw()
line2.draw()

Man mano che definiamo e disegniamo altri oggetti, useremo di nuovo queste linee.
Ma prima parliamo di equivalenza e identità degli oggetti.

## Equivalenza e identità

Supponiamo di creare due punti con le stesse coordinate.

In [ ]:
p1 = Point(200, 100)
p2 = Point(200, 100)

Se usiamo l'operatore `==` per confrontarli, otteniamo il comportamento predefinito per i tipi definiti dal programmatore – il risultato è `True` solo se sono lo stesso oggetto, e non lo sono.

In [ ]:
p1 == p2

Se vogliamo cambiare questo comportamento, possiamo fornire un metodo speciale chiamato `__eq__` che definisce cosa significa che due oggetti `Point` sono uguali.

In [ ]:
%%add_method_to Point

def __eq__(self, other):
    return (self.x == other.x) and (self.y == other.y)

Questa definizione considera uguali due `Point` se i loro attributi sono uguali.
Ora, quando usiamo l'operatore `==`, questo invoca il metodo `__eq__`, che indica che `p1` e `p2` sono considerati uguali.

In [ ]:
p1 == p2

Ma l'operatore `is` indica ancora che sono oggetti diversi.

In [ ]:
p1 is p2

Non è possibile sovrascrivere l'operatore `is` – verifica sempre se gli oggetti sono identici.
Ma per i tipi definiti dal programmatore, puoi sovrascrivere l'operatore `==` in modo che verifichi se gli oggetti sono equivalenti.
E puoi definire cosa significa equivalente.

## Creare un rettangolo

Ora definiamo una classe che rappresenta e disegna rettangoli.
Per semplicità, assumeremo che i rettangoli siano verticali od orizzontali, non inclinati.
Quali attributi pensi che dovremmo usare per specificare la posizione e la dimensione di un rettangolo?

Ci sono almeno due possibilità:

-   Potresti specificare la larghezza e l'altezza del rettangolo e la posizione di un angolo.

-   Potresti specificare due angoli opposti.

A questo punto è difficile dire se una sia migliore dell'altra, quindi implementiamo la prima.
Ecco la definizione della classe.

In [ ]:
class Rectangle:
    """Represents a rectangle. 

    attributes: width, height, corner.
    """
    def __init__(self, width, height, corner):
        self.width = width
        self.height = height
        self.corner = corner
        
    def __str__(self):
        return f'Rectangle({self.width}, {self.height}, {self.corner})'

Come al solito, il metodo `__init__` assegna i parametri agli attributi e `__str__` restituisce una rappresentazione in stringa dell'oggetto.
Ora possiamo istanziare un oggetto `Rectangle`, usando un `Point` come posizione dell'angolo in alto a sinistra.

In [ ]:
corner = Point(30, 20)
box1 = Rectangle(100, 50, corner)
print(box1)

Il diagramma seguente mostra lo stato di questo oggetto.

In [ ]:
from diagram import Binding, Value

def make_rectangle_binding(name, box, **options):
    d1 = vars(box.corner)
    frame_corner = make_frame(d1, name='Point', dy=-0.25, offsetx=0.07)

    d2 = dict(width=box.width, height=box.height)
    frame = make_frame(d2, name='Rectangle', dy=-0.25, offsetx=0.45)
    binding = Binding(Value('corner'), frame1, dx=0.92, draw_value=False, **options)
    frame.bindings.append(binding)

    binding = Binding(Value(name), frame)
    return binding, frame_corner

binding_box1, frame_corner1 = make_rectangle_binding('box1', box1)

In [ ]:
from diagram import Bbox

width, height, x, y = [2.83, 1.49, 0.27, 1.1]
ax = diagram(width, height)
bbox1 = binding_box1.draw(ax, x, y)
bbox2 = frame_corner1.draw(ax, x+1.85, y-0.6)
bbox = Bbox.union([bbox1, bbox2])
#adjust(x, y, bbox)

Per disegnare un rettangolo, useremo il metodo seguente per creare quattro oggetti `Point` che rappresentano gli angoli.

In [ ]:
%%add_method_to Rectangle

    def make_points(self):
        p1 = self.corner
        p2 = p1.translated(self.width, 0)
        p3 = p2.translated(0, self.height)
        p4 = p3.translated(-self.width, 0)
        return p1, p2, p3, p4

Poi creeremo quattro oggetti `Line` che rappresentano i lati.

In [ ]:
%%add_method_to Rectangle

    def make_lines(self):
        p1, p2, p3, p4 = self.make_points()
        return Line(p1, p2), Line(p2, p3), Line(p3, p4), Line(p4, p1)

Poi disegneremo i lati.

In [ ]:
%%add_method_to Rectangle

    def draw(self):
        lines = self.make_lines()
        for line in lines:
            line.draw()

Ecco un esempio.

In [ ]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()

La figura include due linee che rappresentano gli assi.

## Modificare i rettangoli

Consideriamo ora due metodi che modificano i rettangoli, `grow` e `translate`.
Vedremo che `grow` funziona come previsto, ma `translate` ha un bug sottile.
Vedi se riesci a scoprirlo prima che te lo spieghi.

`grow` riceve due numeri, `dwidth` e `dheight`, e li aggiunge agli attributi `width` e `height` del rettangolo.

In [ ]:
%%add_method_to Rectangle

    def grow(self, dwidth, dheight):
        self.width += dwidth
        self.height += dheight

Ecco un esempio che ne mostra l'effetto facendo una copia di `box1` e invocando `grow` sulla copia.

In [ ]:
box2 = copy(box1)
box2.grow(60, 40)
print(box2)

Se disegniamo `box1` e `box2`, possiamo confermare che `grow` funziona come previsto.

In [ ]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()
box2.draw()

Ora vediamo `translate`.
Riceve due numeri, `dx` e `dy`, e sposta il rettangolo delle distanze indicate nelle direzioni `x` e `y`.

In [ ]:
%%add_method_to Rectangle

    def translate(self, dx, dy):
        self.corner.translate(dx, dy)

Per mostrarne l'effetto, trasleremo `box2` verso destra e verso il basso.

In [ ]:
box2.translate(30, 20)
print(box2)

Ora vediamo cosa succede se disegniamo di nuovo `box1` e `box2`.

In [ ]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()
box2.draw()

Sembra che entrambi i rettangoli si siano spostati, e non è ciò che volevamo!
La prossima sezione spiega cosa è andato storto.

## Copia profonda

Quando usiamo `copy` per duplicare `box1`, copia l'oggetto `Rectangle` ma non l'oggetto `Point` che contiene.
Quindi `box1` e `box2` sono oggetti diversi, come previsto.

In [ ]:
box1 is box2

Ma i loro attributi `corner` fanno riferimento allo stesso oggetto.

In [ ]:
box1.corner is box2.corner

Il diagramma seguente mostra lo stato di questi oggetti.

In [ ]:
from diagram import Stack
from copy import deepcopy

binding_box1, frame_corner1 = make_rectangle_binding('box1', box1)
binding_box2, frame_corner2 = make_rectangle_binding('box2', box2, dy=0.4)
binding_box2.value.bindings.reverse()

stack = Stack([binding_box1, binding_box2], dy=-1.3)

In [ ]:
from diagram import Bbox

width, height, x, y = [2.76, 2.54, 0.27, 2.16]
ax = diagram(width, height)
bbox1 = stack.draw(ax, x, y)
bbox2 = frame_corner1.draw(ax, x+1.85, y-0.6)
bbox = Bbox.union([bbox1, bbox2])
# adjust(x, y, bbox)

Quello che fa `copy` si chiama **copia superficiale** (*shallow copy*) perché copia l'oggetto ma non gli oggetti che contiene.
Di conseguenza, cambiare la `width` o la `height` di un `Rectangle` non ha effetto sull'altro, ma cambiare gli attributi del `Point` condiviso ha effetto su entrambi!
Questo comportamento è fonte di confusione e di errori.

Per fortuna, il modulo `copy` fornisce un'altra funzione, chiamata `deepcopy`, che copia non solo l'oggetto ma anche gli oggetti a cui fa riferimento, e gli oggetti a cui fanno riferimento *questi*, e così via.
Questa operazione si chiama **copia profonda** (*deep copy*).

Per dimostrarlo, partiamo da un nuovo `Rectangle` che contiene un nuovo `Point`.

In [ ]:
corner = Point(20, 20)
box3 = Rectangle(100, 50, corner)
print(box3)

E facciamone una copia profonda.

In [ ]:
from copy import deepcopy

box4 = deepcopy(box3)

Possiamo confermare che i due oggetti `Rectangle` fanno riferimento a oggetti `Point` diversi.

In [ ]:
box3.corner is box4.corner

Poiché `box3` e `box4` sono oggetti completamente separati, possiamo modificarne uno senza influire sull'altro.
Per dimostrarlo, sposteremo `box3` e ingrandiremo `box4`.

In [ ]:
box3.translate(50, 30)
box4.grow(100, 60)

E possiamo confermare che l'effetto è quello previsto.

In [ ]:
make_turtle()
line1.draw()
line2.draw()
box3.draw()
box4.draw()

## Polimorfismo

Nell'esempio precedente abbiamo invocato il metodo `draw` su due oggetti `Line` e due oggetti `Rectangle`.
Possiamo fare la stessa cosa in modo più conciso creando una lista di oggetti.

In [ ]:
shapes = [line1, line2, box3, box4]

Gli elementi di questa lista sono di tipi diversi, ma forniscono tutti un metodo `draw`, quindi possiamo scorrere la lista e invocare `draw` su ciascuno.

In [ ]:
make_turtle()

for shape in shapes:
    shape.draw()

Al primo e al secondo giro del ciclo, `shape` fa riferimento a un oggetto `Line`, quindi quando viene invocato `draw`, il metodo eseguito è quello definito nella classe `Line`.

Al terzo e al quarto giro del ciclo, `shape` fa riferimento a un oggetto `Rectangle`, quindi quando viene invocato `draw`, il metodo eseguito è quello definito nella classe `Rectangle`.

In un certo senso, ogni oggetto sa come disegnare sé stesso.
Questa caratteristica si chiama **polimorfismo** (*polymorphism*).
La parola deriva da radici greche che significano "molte forme".
Nella programmazione orientata agli oggetti, il polimorfismo è la capacità di tipi diversi di fornire gli stessi metodi, il che rende possibile eseguire molti calcoli – come disegnare forme – invocando lo stesso metodo su tipi di oggetti diversi.

Come esercizio alla fine di questo capitolo, definirai una nuova classe che rappresenta un cerchio e fornisce un metodo `draw`.
Poi potrai usare il polimorfismo per disegnare linee, rettangoli e cerchi.

## Debugging

In questo capitolo ci siamo imbattuti in un bug sottile, nato dal fatto che abbiamo creato un `Point` condiviso da due oggetti `Rectangle` e poi abbiamo modificato il `Point`.
In generale, ci sono due modi per evitare problemi come questo: puoi evitare di condividere gli oggetti oppure puoi evitare di modificarli.

Per evitare di condividere gli oggetti, puoi usare la copia profonda, come abbiamo fatto in questo capitolo.

Per evitare di modificare gli oggetti, valuta di sostituire le funzioni non pure come `translate` con funzioni pure come `translated`.
Per esempio, ecco una versione di `translated` che crea un nuovo `Point` e non modifica mai i suoi attributi.

In [ ]:
    def translated(self, dx=0, dy=0):
        x = self.x + dx
        y = self.y + dy
        return Point(x, y)

Python fornisce funzionalità che rendono più facile evitare di modificare gli oggetti.
Vanno oltre lo scopo di questo libro, ma se sei curioso chiedi a un assistente virtuale "Come rendo immutabile un oggetto Python?"

Creare un nuovo oggetto richiede più tempo che modificarne uno esistente, ma in pratica la differenza conta raramente.
I programmi che evitano oggetti condivisi e funzioni non pure sono spesso più facili da sviluppare, testare e correggere – e il miglior debugging è quello che non devi fare.

## Glossario

**copia superficiale** (*shallow copy*):
Un'operazione di copia che non copia gli oggetti annidati.

**copia profonda** (*deep copy*):
Un'operazione di copia che copia anche gli oggetti annidati.

**polimorfismo** (*polymorphism*):
La capacità di un metodo o di un operatore di lavorare con più tipi di oggetti.

## Esercizi

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

### Chiedi a un assistente virtuale

Per tutti gli esercizi seguenti, valuta di chiedere aiuto a un assistente virtuale.
Se lo fai, conviene includere nel prompt le definizioni delle classi `Point`, `Line` e `Rectangle` – altrimenti l'assistente farà delle ipotesi sui loro attributi e sulle loro funzioni, e il codice che genera non funzionerà.

### Esercizio

Scrivi un metodo `__eq__` per la classe `Line` che restituisce `True` se gli oggetti `Line` fanno riferimento a oggetti `Point` equivalenti, in qualsiasi ordine.

Puoi usare lo schema seguente per iniziare.

In [ ]:
%%add_method_to Line

def __eq__(self, other):
    return None

In [ ]:
# Solution goes here

Puoi usare questi esempi per testare il tuo codice.

In [ ]:
start1 = Point(0, 0)
start2 = Point(0, 0)
end = Point(200, 100)

Questo esempio dovrebbe dare `True` perché gli oggetti `Line` fanno riferimento a oggetti `Point` equivalenti, nello stesso ordine.

In [ ]:
line_a = Line(start1, end)
line_b = Line(start2, end)
line_a == line_b    # should be True

Questo esempio dovrebbe dare `True` perché gli oggetti `Line` fanno riferimento a oggetti `Point` equivalenti, in ordine inverso.

In [ ]:
line_c = Line(end, start1)
line_a == line_c     # should be True

L'equivalenza dovrebbe essere sempre transitiva – cioè, se `line_a` e `line_b` sono equivalenti, e `line_a` e `line_c` sono equivalenti, allora anche `line_b` e `line_c` dovrebbero essere equivalenti.

In [ ]:
line_b == line_c     # should be True

Questo esempio dovrebbe dare `False` perché gli oggetti `Line` fanno riferimento a oggetti `Point` non equivalenti.

In [ ]:
line_d = Line(start1, start2)
line_a == line_d    # should be False

### Esercizio

Scrivi un metodo di `Line` chiamato `midpoint` che calcola il punto medio di un segmento e restituisce il risultato come oggetto `Point`.

Puoi usare lo schema seguente per iniziare.

In [ ]:
%%add_method_to Line

    def midpoint(self):
        return Point(0, 0)

In [ ]:
# Solution goes here

Puoi usare gli esempi seguenti per testare il tuo codice e disegnare il risultato.

In [ ]:
start = Point(0, 0)
end1 = Point(300, 0)
end2 = Point(0, 150)
line1 = Line(start, end1)
line2 = Line(start, end2)

In [ ]:
mid1 = line1.midpoint()
print(mid1)

In [ ]:
mid2 = line2.midpoint()
print(mid2)

In [ ]:
line3 = Line(mid1, mid2)

In [ ]:
make_turtle()

for shape in [line1, line2, line3]:
    shape.draw()

### Esercizio

Scrivi un metodo di `Rectangle` chiamato `midpoint` che trova il punto al centro di un rettangolo e restituisce il risultato come oggetto `Point`.

Puoi usare lo schema seguente per iniziare.

In [ ]:
%%add_method_to Rectangle

    def midpoint(self):
        return Point(0, 0)

In [ ]:
# Solution goes here

Puoi usare l'esempio seguente per testare il tuo codice.

In [ ]:
corner = Point(30, 20)
rectangle = Rectangle(100, 80, corner)

In [ ]:
mid = rectangle.midpoint()
print(mid)

In [ ]:
diagonal = Line(corner, mid)

In [ ]:
make_turtle()

for shape in [line1, line2, rectangle, diagonal]:
    shape.draw()

### Esercizio

Scrivi un metodo di `Rectangle` chiamato `make_cross` che:

1. Usa `make_lines` per ottenere una lista di oggetti `Line` che rappresentano i quattro lati del rettangolo.

2. Calcola i punti medi delle quattro linee.

3. Crea e restituisce una lista di due oggetti `Line` che rappresentano le linee che collegano i punti medi opposti, formando una croce al centro del rettangolo.

Puoi usare questo schema per iniziare.

In [ ]:
%%add_method_to Rectangle

    def make_diagonals(self):
        return []

In [ ]:
# Solution goes here

Puoi usare l'esempio seguente per testare il tuo codice.

In [ ]:
corner = Point(30, 20)
rectangle = Rectangle(100, 80, corner)

In [ ]:
lines = rectangle.make_cross()

In [ ]:
make_turtle()

rectangle.draw()
for line in lines:
    line.draw()

### Esercizio

Scrivi la definizione di una classe chiamata `Circle` con gli attributi `center` e `radius`, dove `center` è un oggetto Point e `radius` è un numero.
Includi i metodi speciali `__init__` e `__str__`, e un metodo chiamato `draw` che usa le funzioni di `jupyturtle` per disegnare il cerchio.

Puoi usare la funzione seguente, che è una versione della funzione `circle` che abbiamo scritto nel capitolo 4.

In [ ]:
from jupyturtle import make_turtle, forward, left, right
import math
    
def draw_circle(radius):
    circumference = 2 * math.pi * radius
    n = 30
    length = circumference / n
    angle = 360 / n
    left(angle / 2)
    for i in range(n):
        forward(length)
        left(angle)

In [ ]:
# Solution goes here

Puoi usare l'esempio seguente per testare il tuo codice.
Inizieremo con un `Rectangle` quadrato con larghezza e altezza `100`.

In [ ]:
corner = Point(20, 20)
rectangle = Rectangle(100, 100, corner)

Il codice seguente dovrebbe creare un `Circle` che sta all'interno del quadrato.

In [ ]:
center = rectangle.midpoint()
radius = rectangle.height / 2

circle = Circle(center, radius)
print(circle)

Se tutto ha funzionato correttamente, il codice seguente dovrebbe disegnare il cerchio all'interno del quadrato (toccandolo su tutti e quattro i lati).

In [ ]:
make_turtle(delay=0.01)

rectangle.draw()
circle.draw()

[Pensare in Python](https://pensare-in-python.github.io/3a-edizione/) – traduzione italiana non ufficiale di [Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Licenza del codice: [MIT License](https://mit-license.org/)

Licenza del testo (originale e traduzione): [Creative Commons Attribuzione - Non commerciale - Condividi allo stesso modo 4.0 Internazionale](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.it)